In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('train.csv')

In [3]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

X = df.drop(columns=['id', 'accident_risk'])
y = df['accident_risk']

categorical_features = ['road_type', 'lighting', 'weather', 'time_of_day']
numerical_features = ['num_lanes', 'curvature', 'speed_limit', 'num_reported_accidents']
# Boolean columns (True/False) can be treated as numeric (1/0)
bool_features = ['road_signs_present', 'public_road', 'holiday', 'school_season']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_features + bool_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

In [4]:
X_test = df.drop(columns=['id', 'accident_risk'])
y_test = df['accident_risk']

poly_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('poly', PolynomialFeatures(degree=2)),
    ('regressor', LinearRegression())
])

poly_model.fit(X, y)

poly_preds_optuned = poly_model.predict(X_test)
poly_rmse = np.sqrt(mean_squared_error(y_test, poly_preds_optuned))

print(f"Polynomial Regression RMSE: {poly_rmse:.4f}")

Polynomial Regression RMSE: 0.0677


In [5]:
final_df = pd.read_csv('test.csv')

X_final = final_df.drop(columns=['id'])

preds_optuned = poly_model.predict(X_final)

submission = pd.DataFrame({
    'id': final_df['id'],
    'accident_risk': preds_optuned
})

submission.to_csv('submission_poly.csv', index=False)

print("Submission file successfully created: submission.csv")

Submission file successfully created: submission.csv


# Poly RMSE 0.067